In [ ]:
import os
import time

import torch
from InferencePipelines import RepLDMSDXLControlNetPipeline
from .controlnet_preprocess import preprocess_condition


gpu_ids = 0
prompt = "masterpiece, ultra-detailed, winter landscape, traditional Japanese pagoda in foreground, Mount Fuji in the distance, snow-covered rooftops and trees, clear blue sky, natural daylight, realistic lighting, sharp architectural structure, high fidelity, cinematic composition."
condition_img_type = "canny"
condition_img_path = "./condition_imgs/mountain.png"
image_size = (6144, 6144)
init_rates = [0.8]
attn_guidance_scale = 0.005
attn_guidance_density = tuple([0]*31 + [1]*15 + [0]*4)
attn_guidance_decay = None
multi_encoder = True
multi_decoder = True
models_to_cpu = True
show_image = False
random_seed = 42
controlnet_conditioning_scale = 0.5
num_images_per_prompt = 1
cache_dir = "../../huggingface_models"


if __name__ == '__main__':
    assert torch.cuda.is_available()
    device = f"cuda:{gpu_ids}"
    negative_prompt = "blurry, ugly, duplicate, poorly drawn, deformed, mosaic"
    torch.manual_seed(random_seed)
    generator = torch.Generator(device)
    # preprocess the control image
    condition_img, controlnet = preprocess_condition(
        cache_dir=cache_dir, device=device,
        condition_img_path=condition_img_path, condition_img_type=condition_img_type,
        height=image_size[0], width=image_size[1],
    )
    # get models
    pipe = RepLDMSDXLControlNetPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        controlnet=controlnet,
        torch_dtype=torch.float16,
        variant="fp16",
        local_files_only=True,
        cache_dir=cache_dir,
    ).to(device)
    # make save dir
    save_dir = condition_img_path.split('/')[-1].split('.')[0]
    save_dir = os.path.join("./" + save_dir, condition_img_type, f"{init_rates}-{attn_guidance_scale}")
    os.makedirs(save_dir, exist_ok=True)
    # infer images
    with torch.no_grad():
        torch.manual_seed(random_seed)
        generator = torch.Generator(device)
        seed = torch.randint(0, int(1e8), (1,)).item()
        generator = generator.manual_seed(seed)
        time_start = time.time()
        images = pipe(
            prompt, negative_prompt=negative_prompt, condition_image=condition_img, generator=generator,
            height=image_size[0], width=image_size[1],
            num_inference_steps=50, guidance_scale = 7.5, controlnet_conditioning_scale=controlnet_conditioning_scale,
            multi_decoder=multi_decoder, show_image=show_image,
            multi_encoder = multi_encoder, models_to_cpu = models_to_cpu,
            num_resample_timesteps = 50,
            init_rates = init_rates,
            attn_type = 'vanilla',
            attn_guidance_scale = attn_guidance_scale,
            attn_guidance_density = attn_guidance_density,
            attn_guidance_decay = attn_guidance_decay,
            power_calibrate = 0,
            attn_guidance_filter = None,
        )
        file_name = f"{os.path.basename(condition_img_path).split(".")[0]}_{seed}.jpg"
        images[-1].save(os.path.join(save_dir, file_name), quality=95)